In [1]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    mutual_info_score
)
from sklearn.model_selection import KFold
import pickle 

import xgboost as xgb




In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hopesb/student-depression-dataset",
    output_dir="./student_depression_dataset")

print("Path to dataset files:", path)




Path to dataset files: ./student_depression_dataset


In [3]:
df= pd.read_csv(path+"/Student Depression Dataset.csv")
df.head()

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,M.Tech,Yes,1.0,1.0,No,0


In [4]:
df.columns=df.columns.str.lower().str.replace(" ", "_")
for col in df.columns:
    if (df[col].dtype=="str"):
        df[col]= df[col].str.lower().str.replace(" ","_")



In [5]:
del df["id"]
del df["profession"]
del df["work_pressure"]
del df["job_satisfaction"]
del df["city"]

In [6]:
df=df.dropna().reset_index(drop=True)
categorical_columns=["gender","sleep_duration","dietary_habits","degree","have_you_ever_had_suicidal_thoughts_?","family_history_of_mental_illness"]
numerical_columns=["age","academic_pressure","cgpa","study_satisfaction","work/study_hours","financial_stress"]
df.head()

,gender,age,academic_pressure,cgpa,study_satisfaction,sleep_duration,dietary_habits,degree,have_you_ever_had_suicidal_thoughts_?,work/study_hours,financial_stress,family_history_of_mental_illness,depression
0,male,33.0,5.0,8.97,2.0,5-6_hours,healthy,b.pharm,yes,3.0,1.0,no,1
1,female,24.0,2.0,5.90,5.0,5-6_hours,moderate,bsc,no,3.0,2.0,yes,0
2,male,31.0,3.0,7.03,5.0,less_than_5_hours,healthy,ba,no,9.0,1.0,yes,0
3,female,28.0,3.0,5.59,2.0,7-8_hours,moderate,bca,yes,4.0,5.0,yes,1
4,female,25.0,4.0,8.13,3.0,5-6_hours,moderate,m.tech,yes,1.0,1.0,no,0


In [7]:
from sklearn.model_selection import train_test_split

df_full_train, df_test=train_test_split(df, test_size=0.2, random_state=5234)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=5234)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [8]:
y_train=df_train["depression"]
y_val=df_val["depression"]
y_test=df_test["depression"]

In [9]:
del df_train["depression"]
del df_val["depression"]
del df_test["depression"]

In [10]:
dv=DictVectorizer(sparse=False)

X_train=dv.fit_transform(df_train.to_dict(orient="records"))
X_val=dv.transform(df_val.to_dict(orient="records"))
X_test=dv.transform(df_test.to_dict(orient="records"))
features = list(dv.get_feature_names_out())
print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

dtrain=xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval=xgb.DMatrix(X_val, label=y_val, feature_names=features)
dtest=xgb.DMatrix(X_test, label=y_test, feature_names=features)



(16738, 49)
(5580, 49)
(16738,)
(5580,)


In [11]:
xgb_params = {
    'eta': 0.05,
    'max_depth': 10,
    'min_child_weight': 5,

    'objective': 'binary:logistic',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=200)

In [12]:
y_pred = model.predict(dval)
roc_auc_score(y_val, y_pred)

0.913653592770539

In [13]:
def train_xgboost(params,dtrain, num_rounds):
    return xgb.train(params, dtrain, num_boost_round=num_rounds)

In [14]:
eta_list=[0.05, 0.09 ,0.1, 0.2, 0.3]

for eta in eta_list:
    xgb_params = {
        'eta': eta,
        'max_depth': 10,
        'min_child_weight': 5,

        'objective': 'binary:logistic',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    model = train_xgboost(xgb_params, dtrain, 200)
    y_pred = model.predict(dval)
    print(f"{eta}:",roc_auc_score(y_val, y_pred))


0.05: 0.913653592770539
0.09: 0.9111153626043094
0.1: 0.9103405778777434
0.2: 0.9049262014719778
0.3: 0.8976279329757644


In [15]:
max_depth_list=[2,3,4,5,7,10,20, 30]

for md in max_depth_list:
    xgb_params = {
        'eta': 0.05,
        'max_depth': md,
        'min_child_weight': 5,

        'objective': 'binary:logistic',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    model = train_xgboost(xgb_params, dtrain, 200)
    y_pred = model.predict(dval)
    print(f"{md}:",roc_auc_score(y_val, y_pred))


2: 0.9223087826237405
3: 0.9224865246150366
4: 0.9219219558939004
5: 0.9214494856062925
7: 0.9185616939723438
10: 0.913653592770539
20: 0.9106002369005948
30: 0.9093918442312043


In [16]:
min_child_weight_list=[2,3,4,5,7,10,20, 30]

for mcw in min_child_weight_list:
    xgb_params = {
        'eta': 0.05,
        'max_depth': 3,
        'min_child_weight': mcw,

        'objective': 'binary:logistic',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    model = train_xgboost(xgb_params, dtrain, 200)
    y_pred = model.predict(dval)
    print(f"{mcw}:",roc_auc_score(y_val, y_pred))


2: 0.922530178207807
3: 0.9224465310033673
4: 0.9225536020868547
5: 0.9224865246150366
7: 0.9225010314492196
10: 0.9224742137695143
20: 0.9227227597588405
30: 0.9224267671054208


In [17]:
num_rounds=[20,40,70,120,160,200,250,300,400]

for nr in num_rounds:
    xgb_params = {
        'eta': 0.05,
        'max_depth': 3,
        'min_child_weight': 4,

        'objective': 'binary:logistic',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    model = train_xgboost(xgb_params, dtrain, num_rounds=nr)
    y_pred = model.predict(dval)
    print(f"{nr}:",roc_auc_score(y_val, y_pred))


20: 0.8944167986478033
40: 0.904678320933761
70: 0.9159017527982221
120: 0.9209283042974833
160: 0.9220277626202802
200: 0.9225536020868547
250: 0.9227489785325471
300: 0.922751107976097
400: 0.9225396941586702


In [18]:
kfold= KFold(n_splits=5,shuffle=True, random_state=2344)
xgb_params = {
        'eta': 0.05,
        'max_depth': 3,
        'min_child_weight': 4,

        'objective': 'binary:logistic',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

scores=[]

for train_idx, val_idx in kfold.split(df_full_train):
    df_train=df_full_train.iloc[train_idx]
    df_val=df_full_train.iloc[val_idx]
    y_train = df_train.depression.values
    y_val = df_val.depression.values
    df_train=df_train.drop(columns=["depression"])
    df_val=df_val.drop(columns=["depression"])
    X_train=dv.transform(df_train.to_dict(orient="records"))
    X_val=dv.transform(df_val.to_dict(orient="records"))
    dtrain=xgb.DMatrix(X_train, label=y_train, feature_names=features)
    dval=xgb.DMatrix(X_val, label=y_val, feature_names=features)
    model=train_xgboost(dtrain=dtrain, params=xgb_params,num_rounds=250 )
    y_pred=model.predict(dval)
    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)



In [19]:
scores

[0.9209010701232616,
 0.9215155199322936,
 0.9187883022397928,
 0.9158116707309408,
 0.9235369647065645]

In [20]:
xgb_params = {
        'eta': 0.05,
        'max_depth': 3,
        'min_child_weight': 4,

        'objective': 'binary:logistic',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

X_train=dv.transform(df_full_train.to_dict(orient="records"))
y_train=df_full_train["depression"]
dtrain=xgb.DMatrix(X_train, label=y_train, feature_names=features)
final_model=train_xgboost(dtrain=dtrain, params=xgb_params,num_rounds=250)
y_pred=model.predict(dtest)
auc = roc_auc_score(y_test, y_pred)



In [21]:

y_pred_class = (y_pred >= 0.5).astype(int)


accuracy = accuracy_score(y_test, y_pred_class)
recall = recall_score(y_test, y_pred_class)
f1 = f1_score(y_test, y_pred_class)
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_class)


print("Accuracy:  ", round(accuracy, 4))
print("Recall:    ", round(recall, 4))
print("F1 Score:  ", round(f1, 4))
print("AUC:       ", round(auc, 4))
print("Precision: ", round(precision, 4))

Accuracy:   0.8579
Recall:     0.8946
F1 Score:   0.8801
AUC:        0.9241
Precision:  0.8661


In [22]:
with open("xg_boost_model.bin", "wb") as fout:
    pickle.dump((dv, final_model), fout)

In [25]:
dict=df_full_train.to_dict(orient="records")
dict[1]

{'gender': 'female',
 'age': 31.0,
 'academic_pressure': 4.0,
 'cgpa': 5.79,
 'study_satisfaction': 4.0,
 'sleep_duration': 'more_than_8_hours',
 'dietary_habits': 'moderate',
 'degree': 'm.pharm',
 'have_you_ever_had_suicidal_thoughts_?': 'yes',
 'work/study_hours': 6.0,
 'financial_stress': 5.0,
 'family_history_of_mental_illness': 'no',
 'depression': 1}